# TUẦN 2: THỰC NGHIỆM VISION TRANSFORMER (ViT) TRÊN NỀN TẢNG CPU (TÍCH HỢP OpenMP)
**Thành viên thực hiện:** An (CPU & OpenMP)

**Mục tiêu:**
1. Lập trình và tối ưu hóa mô hình Vision Transformer (ViT) trên nền tảng CPU.
2. Tích hợp và đánh giá hiệu năng đa luồng OpenMP (`torch.set_num_threads`, `OMP_NUM_THREADS`).
3. Chạy thực nghiệm huấn luyện và đánh giá trên tập dữ liệu PathMNIST (đồng bộ kiến trúc với TPU của Lộc và GPU của Tuấn).
4. Ghi log các thông số (FLOPS, Latency, Throughput, Lượng RAM tiêu thụ, Speedup) bàn giao cho Hằng làm báo cáo đối sánh phần cứng.

In [ ]:
# CELL 1: Cài đặt và nạp các thư viện cần thiết
import os
import sys
import json
import time
import random
import platform
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PathMNIST
from tqdm.auto import tqdm

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Platform:", platform.platform())
print("CPU Model:", platform.processor())
print(f"Physical Cores: {psutil.cpu_count(logical=False)} | Logical Cores: {psutil.cpu_count(logical=True)}")
print(f"Total RAM: {round(psutil.virtual_memory().total / (1024**3), 2)} GB")

In [ ]:
# CELL 2: Kiểm tra backend OpenMP và cấu hình luồng CPU
device = torch.device("cpu")
print("Device đang sử dụng:", device)
print("Số luồng tính toán OpenMP hiện tại:", torch.get_num_threads())
print("Số luồng inter-op hiện tại:", torch.get_num_interop_threads())

# Kiểm tra phép nhân ma trận trên CPU bằng OpenMP
a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)
start = time.perf_counter()
c = torch.matmul(a, b)
elapsed = (time.perf_counter() - start) * 1000
print(f"Kiểm tra SGEMM 2048x2048 trên CPU: {elapsed:.2f} ms - Thành công!")

In [ ]:
# CELL 3: Cấu hình Hyperparameters (Chuẩn hóa đồng bộ 100% với TPU và GPU)
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

CONFIG = {
    "image_size": 56,
    "patch_size": 4,
    "num_classes": 9,
    "embed_dim": 192,
    "num_heads": 3,
    "depth": 6,
    "mlp_ratio": 4,
    "dropout": 0.0,

    "batch_size": 64,
    "epochs": 5,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,

    "warmup": 10,
    "benchmark_iterations": 30,
    "num_workers": 0,
    "platform": "CPU (OpenMP)"
}

os.makedirs("results", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

with open("results/config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=4)

print(json.dumps(CONFIG, indent=4))

In [ ]:
# CELL 4: Tải và tiền xử lý tập dữ liệu PathMNIST
transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

data_dir = os.path.join(os.path.expanduser("~"), ".medmnist")
train_dataset = PathMNIST(split="train", transform=transform, download=True, as_rgb=True, root=data_dir)
val_dataset = PathMNIST(split="val", transform=transform, download=True, as_rgb=True, root=data_dir)
test_dataset = PathMNIST(split="test", transform=transform, download=True, as_rgb=True, root=data_dir)

print("Số mẫu Train:", len(train_dataset))
print("Số mẫu Validation:", len(val_dataset))
print("Số mẫu Test:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

In [ ]:
# CELL 5: Định nghĩa kiến trúc Vision Transformer (ViT)
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=56, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        assert image_size % patch_size == 0
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim=192, num_heads=3):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.projection = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attention = torch.softmax(scores, dim=-1)
        out = torch.matmul(attention, v).transpose(1, 2).contiguous().reshape(B, N, C)
        return self.projection(out)

class MLP(nn.Module):
    def __init__(self, dim=192, hidden_dim=768):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim=192, num_heads=3, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = MultiHeadSelfAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, dim * mlp_ratio)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, image_size=56, patch_size=4, num_classes=9, embed_dim=192, num_heads=3, depth=6, mlp_ratio=4):
        super().__init__()
        self.patch_embedding = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embedding.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.encoder = nn.ModuleList([TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embedding(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embedding
        for block in self.encoder:
            x = block(x)
        x = self.norm(x[:, 0])
        return self.head(x)

In [ ]:
# CELL 6: Khởi tạo mô hình và kiểm tra Forward Pass trên CPU
model = VisionTransformer(
    image_size=CONFIG["image_size"],
    patch_size=CONFIG["patch_size"],
    num_classes=CONFIG["num_classes"],
    embed_dim=CONFIG["embed_dim"],
    num_heads=CONFIG["num_heads"],
    depth=CONFIG["depth"],
    mlp_ratio=CONFIG["mlp_ratio"]
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Tổng số tham số: {total_params:,}")

# Kiểm tra Forward pass với batch đầu tiên
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)
model.eval()
with torch.no_grad():
    logits = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", logits.shape)
assert logits.shape == (sample_images.shape[0], 9)
assert torch.isfinite(logits).all().item()
print("ViT CPU forward test: THÀNH CÔNG!")

In [ ]:
# CELL 7: Hàm đánh giá trên tập dữ liệu
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.reshape(-1).long().to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
# CELL 8: Huấn luyện mô hình ViT trên CPU với OpenMP
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

# Kiểm tra nếu có sẵn checkpoint đã huấn luyện từ Colab thì nạp vào
CHECKPOINT_PATH = "checkpoints/vit_pathmnist_cpu.pt"
history = []

if os.path.exists(CHECKPOINT_PATH):
    print(f"Nạp trọng số từ checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
else:
    print(f"Bắt đầu huấn luyện {CONFIG['epochs']} epoch trên CPU...")
    for epoch in range(CONFIG["epochs"]):
        model.train()
        epoch_loss = 0.0
        total_samples = 0
        start_time = time.perf_counter()

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}"):
            images = images.to(device)
            labels = labels.reshape(-1).long().to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)

        epoch_time = time.perf_counter() - start_time
        train_loss = epoch_loss / total_samples
        val_loss, val_acc = evaluate(val_loader)

        res = {
            "epoch": epoch + 1,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "val_accuracy": round(val_acc, 4),
            "epoch_time_s": round(epoch_time, 2)
        }
        history.append(res)
        print(res)

    torch.save({"model_state_dict": model.state_dict(), "history": history}, CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv("results/training_history.csv", index=False)

In [ ]:
# CELL 9: Đánh giá chi tiết trên tập Kiểm thử (Test Set)
test_loss, test_accuracy = evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")

pd.DataFrame([{"test_loss": test_loss, "test_accuracy": test_accuracy}]).to_csv(
    "results/test_metrics.csv", index=False
)

In [ ]:
# CELL 10: Benchmark suy luận (Inference) trên CPU qua các Batch Size [1, 8, 32, 64]
def make_fixed_batch(batch_size):
    x, _ = next(iter(test_loader))
    if len(x) < batch_size:
        repeat = (batch_size + len(x) - 1) // len(x)
        x = torch.cat([x] * repeat, dim=0)
    return x[:batch_size]

def benchmark_inference(batch_size, iterations=30, warmup=10):
    model.eval()
    host_x = make_fixed_batch(batch_size)

    # First call (cold start)
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(host_x)
    first_call_ms = (time.perf_counter() - start) * 1000

    # Warm-up
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(host_x)

    # 1. Compute-only
    compute_times = []
    with torch.no_grad():
        for _ in range(iterations):
            start = time.perf_counter()
            _ = model(host_x)
            compute_times.append((time.perf_counter() - start) * 1000)

    # 2. Transfer + Compute
    transfer_times = []
    with torch.no_grad():
        for _ in range(iterations):
            start = time.perf_counter()
            inp = host_x.clone()
            _ = model(inp)
            transfer_times.append((time.perf_counter() - start) * 1000)

    compute_median = float(np.median(compute_times))
    transfer_median = float(np.median(transfer_times))
    mem_info = psutil.Process().memory_info()

    return {
        "batch_size": batch_size,
        "precision": "FP32",
        "execution": "eager_cpu_openmp",
        "first_call_ms": round(first_call_ms, 3),
        "compute_median_ms": round(compute_median, 3),
        "compute_p25_ms": round(float(np.percentile(compute_times, 25)), 3),
        "compute_p75_ms": round(float(np.percentile(compute_times, 75)), 3),
        "compute_throughput": round(batch_size * 1000 / compute_median, 2),
        "transfer_compute_median_ms": round(transfer_median, 3),
        "transfer_compute_throughput": round(batch_size * 1000 / transfer_median, 2),
        "ram_rss_mb": round(mem_info.rss / (1024 * 1024), 2)
    }

BATCH_SIZES = [1, 8, 32, 64]
benchmark_results = []

for bs in BATCH_SIZES:
    res = benchmark_inference(batch_size=bs, iterations=CONFIG["benchmark_iterations"], warmup=CONFIG["warmup"])
    benchmark_results.append(res)
    print(res)

benchmark_df = pd.DataFrame(benchmark_results)
benchmark_df.to_csv("results/cpu_benchmark.csv", index=False)
display(benchmark_df)

In [ ]:
# CELL 11: Thực nghiệm Đa luồng OpenMP Thread Scaling (Khảo sát Speedup & Amdahl's Law)
def benchmark_thread_scaling(batch_size=32, thread_counts=[1, 2, 4, 6, 8, 12]):
    scaling_results = []
    base_median = None

    for t in thread_counts:
        torch.set_num_threads(t)
        res = benchmark_inference(batch_size=batch_size, iterations=20, warmup=5)
        median_ms = res["compute_median_ms"]
        throughput = res["compute_throughput"]

        if base_median is None:
            base_median = median_ms

        speedup = round(base_median / median_ms, 2)
        efficiency = round(speedup / t, 3)

        scaling_results.append({
            "num_threads": t,
            "batch_size": batch_size,
            "median_ms": median_ms,
            "throughput_imgs_per_sec": throughput,
            "speedup": speedup,
            "parallel_efficiency": efficiency,
            "ram_rss_mb": res["ram_rss_mb"]
        })

    return pd.DataFrame(scaling_results)

scaling_df = benchmark_thread_scaling(batch_size=32, thread_counts=[1, 2, 4, 6, 8, 12])
scaling_df.to_csv("results/openmp_thread_scaling.csv", index=False)
display(scaling_df)

# Trực quan hóa Speedup
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(scaling_df["num_threads"], scaling_df["speedup"], marker='o', linewidth=2, label="Đo đạc thực tế")
plt.plot(scaling_df["num_threads"], scaling_df["num_threads"], '--', color='gray', label="Lý tưởng (Tuyến tính)")
plt.xlabel("Số luồng OpenMP")
plt.ylabel("Gia tốc (Speedup)")
plt.title("OpenMP Speedup theo số luồng")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(scaling_df["num_threads"], scaling_df["parallel_efficiency"] * 100, marker='s', color='orange', linewidth=2)
plt.xlabel("Số luồng OpenMP")
plt.ylabel("Hiệu suất song song (%)")
plt.title("Parallel Efficiency theo số luồng")
plt.grid(True)

plt.tight_layout()
plt.savefig("results/openmp_scaling_plot.png", dpi=300)
plt.show()

In [ ]:
# CELL 12: Bảng thống kê log bàn giao cho Hằng (FLOPS, RAM, Throughput)
summary_records = []
for _, row in benchmark_df.iterrows():
    # Tính toán GFLOPS theo công thức ViT
    gflops_per_img = 1.235
    batch_gflops = round(gflops_per_img * row["batch_size"], 4)
    achieved_gflops = round(batch_gflops / (row["compute_median_ms"] / 1000.0), 2)

    summary_records.append({
        "Platform": "CPU (AMD 6C/12T)",
        "Parallel_Framework": "OpenMP",
        "Batch_Size": int(row["batch_size"]),
        "Latency_ms": row["compute_median_ms"],
        "Throughput_img_s": row["compute_throughput"],
        "RAM_RSS_MB": row["ram_rss_mb"],
        "Batch_GFLOPS": batch_gflops,
        "Achieved_GFLOPS": achieved_gflops
    })

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv("results/cpu_memory_flops.csv", index=False)
print("=== BẢNG SỐ LIỆU TỔNG KẾT BÀN GIAO CHO HẰNG ===")
display(summary_df)